# 📋 TV4: Trạng thái & Bàn giao (Frontend, Security QA, Báo cáo)

Notebook này trả lời 3 câu hỏi cho cả nhóm:
1. **TV4 đã làm xong gì?** Có cell chạy kiểm chứng ngay, không cần tin lời.
2. **Còn thiếu gì?** Những thứ chưa xong, không được trình bày như đã xong.
3. **TV4 đang chờ gì?** Tự dò trực tiếp từ code trên repo.

▶️ Chạy **Run All** để cập nhật kết quả kiểm chứng (cần Python + `pip install -r tests/security/requirements.txt`).

---
## 1. ✅ Đã xong

| Hạng mục | Vị trí | Kiểm chứng |
|---|---|---|
| Web App: Đăng nhập Entra/OIDC (MSAL, PKCE), Chuyển Org, Dự án, Task Board, Quản lý Member, Audit History, Export CSV + tải qua SAS | `frontend/` | Cell 1.2 (type-check) |
| Automated API Security Tests 1–3 (pytest) | `tests/security/` | Cell 1.1 (selfcheck); CI của nhóm chạy job TV4 mỗi lần push |
| Ghi bằng chứng: input, timestamp, raw HTTP, tự che JWT/SAS/password | `tests/security/evidence.py` | `test_0_evidence_redaction.py` (trong 1.1) |
| Máy quét rò rỉ bí mật trước khi nộp | `tests/security/scan_secrets.py` | Cell 1.3 |
| Mock API theo contract (chỉ dùng cho dev/QA) | `tests/mock_api/server.py` | `API_CONTRACT_PROPOSAL.ipynb` phần B |
| Design Report (đã điền số liệu có sẵn đến 23/11) | `docs/tv4/DESIGN_REPORT.ipynb` | Còn chỗ `⟦ĐIỀN⟧` chờ số đo trên Azure |
| Kịch bản Demo 10–15 phút | `docs/tv4/DEMO_SCRIPT.ipynb` | — |
| API Contract đề xuất | `docs/tv4/API_CONTRACT_PROPOSAL.ipynb` | **Chờ TV2 xác nhận** |

In [1]:
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists() or (p / "infra").is_dir())
print("Repo root:", ROOT.name)

Repo root: multi-tenant-project-management


### 1.1 Selfcheck: bộ test có thực sự bắt được lỗi không?

Chạy bộ test 6 lần trên mock API:
- 1 lần với mock **an toàn** → tất cả phải **PASS**;
- 5 lần, mỗi lần mock **cố tình có 1 lỗ hổng** → test tương ứng phải **FAIL**.

| `--vuln` | Lỗ hổng giả lập | Test phải bắt |
|---|---|---|
| `idor` | API không kiểm tra tài nguyên thuộc tenant nào | Test 1 |
| `stale-membership` | API vẫn cho user đã bị thu hồi truy cập | Test 2 |
| `global-unique` | `UNIQUE(name)` toàn cục thay vì `(tenant_id, name)` | Test 3 |
| `shared-cache` | Cache key không có `tenant_id` | Test 3 |
| `demo-header` | Header `X-Demo-User` được nhận thay cho JWT (chế độ demo của backend còn bật) | Test 1 |

In [2]:
import subprocess, sys, os
res = subprocess.run([sys.executable, str(ROOT / "tests" / "security" / "selfcheck.py")],
                     capture_output=True, text=True, encoding="utf-8", env={**os.environ, "PYTHONIOENCODING": "utf-8"})
print(res.stdout.strip() or res.stderr[-2000:])

[OK ] secure mock        -> tests must PASS : 36 passed in 2.14s
[OK ] --vuln idor             -> test_1_cross_tenant_idor.py must FAIL : 15 failed, 14 passed in 1.43s
[OK ] --vuln stale-membership -> test_2_membership_revocation.py must FAIL : 1 failed, 1 passed in 0.45s
[OK ] --vuln global-unique    -> test_3_duplicate_name_cache.py must FAIL : 1 failed in 0.34s
[OK ] --vuln shared-cache     -> test_3_duplicate_name_cache.py must FAIL : 1 failed in 0.49s
[OK ] --vuln demo-header      -> test_1_cross_tenant_idor.py must FAIL : 2 failed, 27 passed in 1.71s

Selfcheck: 6/6 scenarios behaved as expected


### 1.2 Frontend type-check
Cần `npm ci` trong `frontend/` trước. Nếu chưa cài, cell sẽ bỏ qua.

In [3]:
import shutil
fe = ROOT / "frontend"
npx = shutil.which("npx")
if not (fe / "node_modules").exists() or not npx:
    print("⏸️ Bỏ qua: chạy `cd frontend && npm ci` trước")
else:
    r = subprocess.run([npx, "tsc", "-b", "--noEmit"], cwd=fe, capture_output=True, text=True, encoding="utf-8")
    print("✅ TypeScript strict: 0 lỗi" if r.returncode == 0 else r.stdout[-2000:])
    for f in fe.glob("*.tsbuildinfo"):
        f.unlink()

✅ TypeScript strict: 0 lỗi


### 1.3 Quét bí mật trong code TV4 (JWT, SAS, AccountKey, password)

In [4]:
targets = [ROOT / "frontend" / "src", ROOT / "frontend" / ".env.example", ROOT / "docs" / "tv4", ROOT / "tests"]
r = subprocess.run([sys.executable, str(ROOT / "tests" / "security" / "scan_secrets.py"), *map(str, targets)],
                   capture_output=True, text=True, encoding="utf-8", env={**os.environ, "PYTHONIOENCODING": "utf-8"})
print(("✅ " if r.returncode == 0 else "❌ ") + r.stdout.strip().splitlines()[-1].split(":")[-1].strip())

✅ 0 nghi vấn


---
## 2. ⏳ Chưa làm được (cập nhật 23/11/2026)

| Việc | Vì sao chưa | Khi nào |
|---|---|---|
| Ghép frontend + Test 1–3 với **backend thật** | Backend TV2 (`backend-core-api/`) đã vào repo nhưng mới có CRUD `workspaces` + export; chưa có `/me`, members, task, audit list, export list (xem cell 3) | Khi các endpoint này có |
| Đăng nhập Entra **thật** | Cần tenant External ID + 2 app registration (SPA + API) | Trước Azure Đợt 1 |
| Bằng chứng Azure trong `docs/tv4/evidence/` | Chỉ thu được khi bật Azure | Đợt 2, Tuần 6 |
| Kiểm tra UI bằng mắt trên trình duyệt | Phiên phát triển chỉ kiểm qua build, type-check và gọi HTTP | Demo nội bộ |

Mọi kết quả PASS của Security Test hiện tại là trên **mock API**, chưa phải backend thật.

---
## 3. 🔎 TV4 đang chờ gì (tự dò từ code trên repo)

Cell dưới đọc **trực tiếp** Terraform, workflow CD và controller của backend để biết các điều kiện mà frontend và Security Test cần đã có chưa.
Dòng comment trong file được **bỏ qua**, để một ghi chú kiểu `# ... = false` không bị tính là cấu hình thật.

In [5]:
import re

def code_only(text):
    """Bỏ comment (#, //, /* */) để chỉ xét cấu hình/code thật."""
    text = re.sub(r"/\*.*?\*/", "", text, flags=re.S)
    return "\n".join(re.split(r"\s(#|//)|^(#|//)", line, maxsplit=1)[0] for line in text.splitlines())

def read_all(folder, pattern):
    return "\n".join(code_only(p.read_text(encoding="utf-8")) for p in (ROOT / folder).rglob(pattern))

tf = read_all("infra", "*.tf")
storage = code_only((ROOT / "infra" / "storage_queue.tf").read_text(encoding="utf-8"))
cd = code_only((ROOT / ".github" / "workflows" / "deploy-and-rollback.yml").read_text(encoding="utf-8"))
api = read_all("backend-core-api/src/main/java", "*Controller.java") if (ROOT / "backend-core-api").exists() else ""

checks = [
    ("Hạ tầng", "Static Web App cho frontend",           bool(re.search(r'resource\s+"azurerm_static_web_app"', tf))),
    ("Hạ tầng", "CORS cho API",                          bool(re.search(r"\bcors\s*\{", tf))),
    ("Hạ tầng", "Tắt demo mode trên Azure",              bool(re.search(r'"APP_SECURITY_DEMO_MODE"\s*=\s*"false"', tf))),
    ("Hạ tầng", "Alert tenant-mismatch cho Failure Drill", "worker_tenant_mismatch_total" in tf),
    ("Hạ tầng", "Storage tắt public network access",     bool(re.search(r"public_network_access_enabled\s*=\s*false", storage))),
    ("CD",      "Workflow CD deploy frontend lên SWA",   bool(re.search(r"static-web-apps-deploy|npm run build", cd))),
    ("Backend", "GET /me (danh sách org của user)",      bool(re.search(r'Mapping\(\s*"[^"]*/me"', api))),
    ("Backend", "Quản lý thành viên",                    bool(re.search(r'Mapping\(\s*"[^"]*members', api))),
    ("Backend", "CRUD task",                             bool(re.search(r'Mapping\(\s*"[^"]*tasks', api))),
    ("Backend", "Danh sách audit",                       bool(re.search(r'Mapping\(\s*"[^"]*audit', api))),
    ("Backend", "Tải export qua SAS",                    bool(re.search(r'Mapping\(\s*"[^"]*/download', api))),
]
for area, item, ok in checks:
    print(f"{'✅ có  ' if ok else '⏳ chưa'} [{area:7}] {item}")

✅ có   [Hạ tầng] Static Web App cho frontend
✅ có   [Hạ tầng] CORS cho API
✅ có   [Hạ tầng] Tắt demo mode trên Azure
✅ có   [Hạ tầng] Alert tenant-mismatch cho Failure Drill
⏳ chưa [Hạ tầng] Storage tắt public network access
⏳ chưa [CD     ] Workflow CD deploy frontend lên SWA
⏳ chưa [Backend] GET /me (danh sách org của user)
⏳ chưa [Backend] Quản lý thành viên
⏳ chưa [Backend] CRUD task
⏳ chưa [Backend] Danh sách audit
✅ có   [Backend] Tải export qua SAS


---
## 4. 🗓️ Kế hoạch Đợt 2 (Tuần 6) của TV4

| Bước | Lệnh / Việc |
|---|---|
| 1 | Lấy token mới cho 4 user (xem `tests/security/README.md`) → `tests/security/tokens.local.json` |
| 2 | `.\tests\security\run.ps1 -Target azure -BaseUrl <api_https_url>` |
| 3 | `python tests/security/scan_secrets.py tests/security/evidence` → copy bản sạch vào `docs/tv4/evidence/<run>/` → PR |
| 4 | Chụp UI: Guest bị chặn, User X bị 403 sau khi thu hồi, Export tải qua SAS |
| 5 | Điền các chỗ `⟦ĐIỀN⟧` trong `DESIGN_REPORT.ipynb` (§4 sẽ tự đọc kết quả từ `docs/tv4/evidence/`) |